In [23]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

# 1. Cargar y reproyectar a WGS84 (necesario para Leaflet/Folium)
gdf = gpd.read_file("edificios_riesgo_cadiz.gpkg").to_crs(epsg=4326)

# 2. Crear el mapa
m = folium.Map(location=[36.5298, -6.2927], zoom_start=14, tiles="CartoDB positron")

color_map = {
    "Riesgo Alto (<= 2m)": "#e74c3c",
    "Riesgo Medio (2m - 5m)": "#f39c12",
    "Riesgo Bajo (> 5m)": "#2ecc71",
    "Sin datos": "#95a5a6"
}

# 3. Capa GeoJSON interactiva
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        'fillColor': color_map.get(feature['properties']['riesgo_inundacion'], '#3186cc'),
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['altitud_min', 'altitud_media', 'riesgo_inundacion'],
        aliases=['Cota Mínima (m):', 'Cota Media (m):', 'Riesgo:'],
        localize=True
    )
).add_to(m)

# 4. Guardar como index.html
m.save("index.html")
print("¡index.html generado!")

# 1. Cargar datos vectoriales
edificios = gpd.read_file("edificios_cadiz_clip.gpkg")
vias = gpd.read_file("vias_cadiz_clip.gpkg")

# 2. Cargar datos ráster
mdt = rasterio.open("mdt_cadiz.tif")

print(f"Edificios cargados: {len(edificios)} filas.")
print(f"Sistema de referencia vectorial: {edificios.crs}")
print(f"Dimensiones del ráster MDT: {mdt.width}x{mdt.height} píxeles.")

¡index.html generado!
Edificios cargados: 5163 filas.
Sistema de referencia vectorial: EPSG:25830
Dimensiones del ráster MDT: 4022x5688 píxeles.


In [24]:
# 1. Calcular estadísticas de altitud para cada geometría de edificio
stats_edificios = zonal_stats( #Cruzo espacialmente los argumentos: edificios y mdt.
    edificios,
    "mdt_cadiz.tif",
    stats=["mean", "min", "max"],
    geojson_out=False
)

# 2. Asignar los resultados como nuevas columnas en el GeoDataFrame
edificios["altitud_media"] = [s["mean"] for s in stats_edificios]
edificios["altitud_min"] = [s["min"] for s in stats_edificios]
edificios["altitud_max"] = [s["max"] for s in stats_edificios]

# 3. Comprobar los primeros resultados
edificios[["altitud_media", "altitud_min", "altitud_max"]].head()

,altitud_media,altitud_min,altitud_max
0,NaN,NaN,NaN
1,17.860987,15.990,18.700001
2,5.298438,4.270,5.620000
3,12.283869,11.694,12.880000
4,4.989035,3.960,5.839000


In [25]:
# Definir categorías de riesgo basadas en la altitud mínima del cimiento (min)
def clasificar_riesgo(alt_min):
    if alt_min is None:
        return "Sin datos"
    elif alt_min <= 2.0:
        return "Riesgo Alto (<= 2m)"
    elif alt_min <= 5.0:
        return "Riesgo Medio (2m - 5m)"
    else:
        return "Riesgo Bajo (> 5m)"

# Aplicar la función a la columna de altitud mínima
edificios["riesgo_inundacion"] = edificios["altitud_min"].apply(clasificar_riesgo)

# Mostrar el recuento de edificios en cada categoría
print(edificios["riesgo_inundacion"].value_counts())

riesgo_inundacion
Riesgo Bajo (> 5m)        2904
Riesgo Medio (2m - 5m)    2177
Riesgo Alto (<= 2m)         82
Name: count, dtype: int64


In [26]:
# Exportar a GeoPackage con las nuevas columnas. Este mapa se usara en QGis
edificios.to_file("edificios_riesgo_cadiz.gpkg", driver="GPKG")
print("¡Archivo exportado correctamente!")

¡Archivo exportado correctamente!


In [27]:
import folium
from branca.element import Template, MacroElement

# 1. Cargar datos en WGS84
gdf = gpd.read_file("edificios_riesgo_cadiz.gpkg").to_crs(epsg=4326)

# 2. Crear mapa
m = folium.Map(tiles="CartoDB positron")

# 3. Definir paleta de colores
color_map = {
    "Riesgo Alto (<= 2m)": "#e74c3c",
    "Riesgo Medio (2m - 5m)": "#f39c12",
    "Riesgo Bajo (> 5m)": "#2ecc71",
    "Sin datos": "#95a5a6"
}

# 4. Añadir capa GeoJSON
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        'fillColor': color_map.get(feature['properties']['riesgo_inundacion'], '#3186cc'),
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['altitud_min', 'altitud_media', 'riesgo_inundacion'],
        aliases=['Cota Mínima (m):', 'Cota Media (m):', 'Nivel de Riesgo:'],
        localize=True
    )
).add_to(m)

# 5. Ajustar encuadre automático
bounds = gdf.total_bounds
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# 6. HTML/CSS para la leyenda flotante
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 210px; height: auto; 
    background-color: white; z-index:9999; font-size:13px;
    border:2px solid grey; border-radius:8px; padding: 10px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    ">
    <b style="font-size:14px;">Riesgo de Inundación</b><br>
    <small>Cota sobre el nivel del mar</small><hr style="margin: 5px 0;">
    <i style="background:#e74c3c; width:15px; height:15px; float:left; margin-right:8px; opacity:0.8; border-radius:3px;"></i> Riesgo Alto (&le; 2m)<br>
    <i style="background:#f39c12; width:15px; height:15px; float:left; margin-right:8px; opacity:0.8; border-radius:3px;"></i> Riesgo Medio (2m - 5m)<br>
    <i style="background:#2ecc71; width:15px; height:15px; float:left; margin-right:8px; opacity:0.8; border-radius:3px;"></i> Riesgo Bajo (&gt; 5m)<br>
</div>
{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(legend_html)
m.get_root().add_child(macro)

# 7. Guardar como index.html
m.save("index.html")
print("¡index.html generado con leyenda flotante!")

¡index.html generado con leyenda flotante!
